## Exploração inicial — Acessos de Banda Larga Fixa (Anatel)

Objetivo: conhecer a base de acessos de banda larga fixa (SCM) da Anatel antes de construir o pipeline de dados. Aqui confirmo formato, colunas, volume e as decisões que orientam o ETL.

**Fonte:** [dados abertos da Anatel](https://www.anatel.gov.br/dadosabertos/paineis_de_dados/acessos/acessos_banda_larga_fixa.zip), baixados com `src/extract.py`.

### 1. Arquivos disponíveis

O zip tem cerca de 1 GB e, descompactado, ocupa uns 10 GB em 26 arquivos CSV:

- `Acessos_Banda_Larga_Fixa_AAAA.csv`: a base principal, com uma linha por mês, empresa, município e característica do acesso. **É a base usada no projeto.**
- `..._Colunas.csv`: os mesmos dados em formato largo, com um mês por coluna. Não será usado.
- `Densidade_Banda_Larga_Fixa.csv`: acessos por 100 habitantes, por município. Pode virar uma análise complementar.

Atenção: a lista é ordenada alfabeticamente, então os arquivos são escolhidos **pelo nome**, nunca pela posição.

In [ ]:
#1
from pathlib import Path
import pandas as pd 

pd.set_option("display.max_columns", None)

#Funciona tanto rodar na raíz como na pasta notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PASTA = ROOT / "data" / "raw" / "acessos_banda_larga_fixa"

arquivos = sorted(PASTA.rglob("*.csv"))
for a in arquivos:
    print(f"{a.name:55} {a.stat().st_size / 1e6:8.1f} MB")

### 2. O arquivo cru

Antes de usar o pandas, leio as primeiras linhas como texto para confirmar o formato com os próprios olhos:

- separador `;`
- encoding UTF-8, com os acentos corretos
- números decimais com **vírgula** (padrão brasileiro), como em `10,000000`

In [ ]:
#2
arquivo = PASTA / "Acessos_Banda_Larga_Fixa_2026.csv"
print(arquivo.name)
with open(arquivo, encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline())

### 3. Colunas e tipos

A base tem **16 colunas**: período (Ano, Mês), prestadora (Grupo Econômico, Empresa, CNPJ, Porte), localização (UF, Município, Código IBGE), características do acesso (Faixa de Velocidade, Velocidade, Tecnologia, Meio de Acesso, Tipo de Pessoa, Tipo de Produto) e a medida, **Acessos**.

**Cuidado importante:** `CNPJ` e `Código IBGE Município` precisam ser lidos como **texto**. Muitos CNPJs começam com zero e, lidos como número, perderiam esse zero, quebrando filtros e junções.

In [ ]:
#3
amostra = pd.read_csv(arquivo, sep=";", nrows=5000, dtype=str)
print(f"Colunas da tabela: {list(amostra.columns)}")
amostra.head()

### 4. Filtrando Alagoas sem estourar a memória

Só o arquivo de 2026 tem quase 1 GB. Em vez de carregá-lo inteiro, leio em pedaços de 500 mil linhas (`chunksize`) e guardo apenas as linhas de Alagoas.

**Resultado (2026):** 6.355.655 linhas no Brasil, das quais **69.300 são de Alagoas** (cerca de 1%). Essa mesma técnica será usada no `src/transform.py`.

In [ ]:
#4
pedacos = []
total_linhas = 0
for chunk in pd.read_csv(arquivo, sep=";", chunksize=500_000, dtype={"CNPJ": str, "Código IBGE Município":str}):
    total_linhas +=len(chunk)
    pedacos.append(chunk[chunk["UF"] == "AL"])

al = pd.concat(pedacos, ignore_index = True)
print(f"Linhas no arquivo: {total_linhas:,} | Linhas de Alagoas: {len(al):,}")

### 5. Perfil dos dados de Alagoas

- **Período em 2026:** janeiro a setembro
- **Municípios:** 102, todos os do estado
- **Empresas reportando:** 379, um retrato da fragmentação do mercado, com centenas de provedores regionais
- **Tecnologia:** FTTH (fibra até a casa) é a mais frequente

**Regra fundamental:** a coluna `Acessos` é um **estoque mensal**, ou seja, quantos acessos existiam naquele mês. **Nunca se soma Acessos entre meses diferentes**; compara-se um mês com outro.

In [ ]:
#5
for coluna in ["Ano", "Mês", "Tecnologia", "Meio de Acesso",
               "Tipo de Pessoa", "Tipo de Produto", "Porte da Prestadora"]:
    if coluna in al.columns:
        print(f"\n--- {coluna} ---")
        print(al[coluna].value_counts().head(15))

print("\nMunicípios:", al["Município"].nunique())   # Alagoas tem 102
print("Empresas:", al["Empresa"].nunique())

### 6. Quem domina o mercado (setembro de 2026)

Alagoas tinha **387.165 acessos** de banda larga fixa em setembro de 2026.

- A **Brisanet** lidera com cerca de 117 mil acessos (~30%), seguida da **Claro** (~22%).
- Abaixo delas há uma longa cauda de provedores regionais.
- A **Starlink** (satélite) já aparece entre as 15 maiores. Isso sugere demanda em áreas onde a fibra ainda não chegou e merece uma análise própria.

Observação: este ranking soma todos os tipos de produto. Nas análises de mercado do projeto, o filtro será `Tipo de Produto = INTERNET`.

In [ ]:
#6
setembro = al[al["Mês"] == al["Mês"].max()]
print("Total de acessos em AL (set/2026):", f"{setembro['Acessos'].sum():,}")
(setembro.groupby(["Grupo Econômico", "Empresa"])["Acessos"]
         .sum().sort_values(ascending=False).head(15))

### 7. Mudança de metodologia em 2021

Os arquivos até 2020 têm **13 colunas**; a partir de 2021, **16**. Em 2021 a Anatel mudou a coleta: passou a separar meio de acesso e tecnologia e trocou as faixas de velocidade pelas velocidades contratadas.

Comparar dados de antes e depois dessa mudança geraria conclusões erradas, então o projeto usa apenas **janeiro de 2021 a setembro de 2026**.

In [ ]:
#7
for a in sorted(PASTA.glob("Acessos_Banda_Larga_Fixa_*.csv")):
    if "Colunas" in a.name or "Total" in a.name:
        continue
    cols = pd.read_csv(a, sep=";", nrows=1).columns.tolist()
    print(f"{a.name:45} {len(cols)} colunas")

### Conclusões para o pipeline

1. Usar os arquivos anuais de **2021 a 2026** (16 colunas, metodologia consistente).
2. Ler em pedaços (`chunksize`) e filtrar **UF = AL**.
3. Ler `CNPJ` e `Código IBGE Município` como texto, e `Velocidade` com `decimal=","`.
4. Converter Ano e Mês numa data (`ano_mes`, sempre o dia 1) para facilitar as séries temporais.
5. Nas análises de mercado, filtrar `Tipo de Produto = INTERNET`.
6. Nunca somar `Acessos` entre meses diferentes.
7. Limitação: algumas prestadoras reportam de forma intermitente (subnotificação), o que deve ser registrado no README.